# GigShield Premium Risk Scoring Model

**Algorithm:** XGBoost Regression

**Purpose:** Predict a weekly risk score (0.5 to 1.5) for each worker-zone pair, used in the dynamic pricing formula:

```
Weekly Premium = Base(₹25) × Zone Risk Score × Season Factor × Loyalty Discount × Tier Multiplier
```

**Why XGBoost:** Handles mixed tabular features well, trains quickly, proven standard for insurance risk scoring.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import joblib
import os

try:
    import xgboost as xgb
    print(f'XGBoost version: {xgb.__version__}')
except ImportError:
    print('XGBoost not installed. Install with: pip install xgboost')

from feature_engineering import (
    generate_synthetic_zones,
    generate_pricing_training_data,
    prepare_pricing_features
)

plt.style.use('seaborn-v0_8-whitegrid')
SEED = 42
print('Libraries loaded successfully')

## 1. Generate Training Data

In [ ]:
zones_df = generate_synthetic_zones(n_zones=50, seed=SEED)
pricing_df = generate_pricing_training_data(zones_df, n_weeks=52, seed=SEED)

print(f'Training data: {pricing_df.shape}')
print(f'Target range: {pricing_df["weekly_risk_score"].min():.3f} to {pricing_df["weekly_risk_score"].max():.3f}')
pricing_df.describe()

## 2. Exploratory Data Analysis

In [ ]:
# Target distribution
fig, axes = plt.subplots(1, 3, figsize=(16, 4))

axes[0].hist(pricing_df['weekly_risk_score'], bins=40, color='steelblue', edgecolor='white')
axes[0].set_title('Weekly Risk Score Distribution')
axes[0].set_xlabel('Risk Score')

# Zone risk vs weekly risk
axes[1].scatter(pricing_df['zone_risk_score'], pricing_df['weekly_risk_score'], alpha=0.3, s=5)
axes[1].set_xlabel('Zone Risk Score')
axes[1].set_ylabel('Weekly Risk Score')
axes[1].set_title('Zone Risk vs Weekly Risk')

# Monthly boxplot
pricing_df.boxplot(column='weekly_risk_score', by='month', ax=axes[2])
axes[2].set_title('Weekly Risk by Month (Seasonal Pattern)')
axes[2].set_xlabel('Month')
axes[2].set_ylabel('Risk Score')

plt.tight_layout()
plt.show()

## 3. Feature Engineering

In [ ]:
X, y, encoders = prepare_pricing_features(pricing_df)
print(f'Features: {X.shape[1]} columns')
print(f'Feature names: {list(X.columns)}')
X.head()

## 4. Train XGBoost Model

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=SEED)
print(f'Train: {X_train.shape}, Test: {X_test.shape}')

model = xgb.XGBRegressor(
    n_estimators=300,
    max_depth=5,
    learning_rate=0.05,
    reg_alpha=0.1,
    reg_lambda=1.0,
    random_state=SEED,
    n_jobs=-1
)

model.fit(
    X_train, y_train,
    eval_set=[(X_test, y_test)],
    verbose=50
)

print('\nTraining complete!')

## 5. Model Evaluation

In [ ]:
y_pred = model.predict(X_test)

mae = mean_absolute_error(y_test, y_pred)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))
r2 = r2_score(y_test, y_pred)

print(f'MAE:  {mae:.4f}')
print(f'RMSE: {rmse:.4f}')
print(f'R²:   {r2:.4f}')

# Predicted vs Actual scatter
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].scatter(y_test, y_pred, alpha=0.3, s=10)
axes[0].plot([0.5, 1.5], [0.5, 1.5], 'r--', linewidth=2)
axes[0].set_xlabel('Actual Risk Score')
axes[0].set_ylabel('Predicted Risk Score')
axes[0].set_title(f'Predicted vs Actual (R² = {r2:.3f})')

# Feature importance
importance = model.feature_importances_
feat_imp = pd.Series(importance, index=X.columns).sort_values(ascending=True)
feat_imp.plot(kind='barh', ax=axes[1], color='steelblue')
axes[1].set_title('Feature Importance')
axes[1].set_xlabel('Importance Score')

plt.tight_layout()
plt.show()

## 6. Pricing Formula Integration

**Full Formula:**
```
Premium = Base(₹25) × Zone Risk Score × Season Factor × Loyalty Discount × Tier Multiplier
```

**Worked example for Ravi (August, Kukatpally, Standard tier, 8 weeks active):**

In [ ]:
# Ravi's premium calculation
base_premium = 25
zone_risk_score = 0.68  # Kukatpally zone
season_factor = 1.30    # August (monsoon)
loyalty_discount = 0.95 # 8 weeks active
tier_multiplier = 1.4   # Standard tier

premium = base_premium * zone_risk_score * season_factor * loyalty_discount * tier_multiplier

print(f'=== Ravi\'s Premium Calculation ===')
print(f'Base Premium:     ₹{base_premium}')
print(f'Zone Risk Score:  {zone_risk_score} (Kukatpally)')
print(f'Season Factor:    {season_factor} (August/Monsoon)')
print(f'Loyalty Discount: {loyalty_discount} (8 weeks, 5% off)')
print(f'Tier Multiplier:  {tier_multiplier} (Standard)')
print(f'---')
print(f'Weekly Premium:   ₹{premium:.0f}')
print(f'Coverage Limit:   ₹350/event')
print(f'For ₹{premium:.0f}/week, Ravi gets up to ₹700 protection (2 events × ₹350)')

## 7. Save Model

In [ ]:
os.makedirs('models', exist_ok=True)

joblib.dump(model, 'models/pricing_xgboost.pkl')
joblib.dump(encoders, 'models/pricing_encoders.pkl')

print('Model saved to models/pricing_xgboost.pkl')
print('Encoders saved to models/pricing_encoders.pkl')
print(f'Training samples: {len(X_train)}, Test samples: {len(X_test)}')
print(f'Metrics — MAE: {mae:.4f}, RMSE: {rmse:.4f}, R²: {r2:.4f}')